# IndoBERT Token Classifier (Colab)

Notebook ini untuk inferensi label token pantun modern dengan skema label BIO (`B/I-ASG`, `B/I-SKT`, `B/I-DSM`, `B/I-TYP`, `O`) memakai model IndoBERT yang disimpan di Google Drive.

Sesuai dokumentasi pipeline:
- `B/I-SKT` -> Rumus A (fonemisasi alfabetis)
- `B/I-DSM` / `B/I-TYP` -> slang.csv -> g2p-id (fallback direct)
- `B/I-ASG` -> failsafe G2P asing
- `O` -> Cascaded CRF 1 (G2P kata baku)


In [1]:
!pip -q install transformers sentencepiece accelerate g2p-id g2p-en nltk symspellpy sklearn-crfsuite joblib
!pip install -q PySastrawi

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
from google.colab import drive

drive.mount('/content/drive')
# UBAH sesuai lokasi folder model fine-tuned Anda di Google Drive.
# Folder ini minimal berisi: config.json, pytorch_model.bin/model.safetensors,
# tokenizer.json atau vocab.txt + tokenizer config.
MODEL_DIR = '/content/drive/MyDrive/Models/indobert-pantun-ner'  # <- ganti path ini

# Lexicon slang (CSV dengan kolom: slang,formal) dari Google Drive.
SLANG_CSV_PATH = '/content/drive/MyDrive/slang.csv'  # <- ganti bila perlu

# Guard agar koreksi SymSpell untuk TYP/DSM tidak terlalu agresif.
SYMSPELL_TYP_MAX_EDIT_DISTANCE = 1
SYMSPELL_TYP_MAX_LEN_DELTA = 1
SYMSPELL_TYP_REQUIRE_SAME_INITIAL = True
SYMSPELL_TYP_REQUIRE_SAME_FINAL = True

# Rescue salah-label ASG: jika hasil stem ada di kata dasar Sastrawi,
# arahkan ke jalur G2P-ID.

# Rescue salah-label O: jika O low-confidence namun bentuk fonem kuat asing,
# naikkan ke ASG.
O_TO_ASG_CONF_THRESHOLD = 0.65
O_TO_ASG_MARGIN_THRESHOLD = 0.20

# Toggle patch ortografi BI untuk jalur G2P_ID (default OFF = CRF BI murni).
ENABLE_ID_ORTHOGRAPHIC_BI_PATCH = False

# Model CRF layer-3 (syllabifier B/I dari deret fonem hasil G2P).
CRF_BI_MODEL_PATH = '/content/drive/MyDrive/Models/crf_layer3_bi_baru.joblib'  # <- ganti bila perlu


Mounted at /content/drive


In [ ]:
import os
import joblib
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForTokenClassification.from_pretrained(MODEL_DIR).to(device)
model.eval()

id2label = model.config.id2label
label2id = model.config.label2id
print('Labels:', id2label)

# Preload CRF BI di bagian atas agar edit logic helper tidak memicu reload model.
crf_bi_model = None
crf_target = str(globals().get('CRF_BI_MODEL_PATH', '')).strip()
if crf_target and os.path.exists(crf_target):
    crf_bi_model = joblib.load(crf_target)
    print(f'[CRF3] Loaded model: {crf_target}')
else:
    print(f'[CRF3] Model belum dimuat (path kosong/tidak ditemukan): {crf_target}')

In [ ]:
# Setup lexicon slang CSV + SymSpell + DSM skeleton (jalankan sebelum cell helper)
import csv
import os
import re
from symspellpy import SymSpell


def _resolve_slang_csv_path() -> str:
    configured = globals().get('SLANG_CSV_PATH', '/content/drive/MyDrive/slang.csv')
    return str(configured).strip()


def _norm_lexicon_term(text: str) -> str:
    t = ''.join(ch for ch in str(text).lower().strip() if ch.isalnum())
    return re.sub(r"(.)\1{2,}", r"\1\1", t)


def _norm_formal_term(text: str) -> str:
    t = ''.join(ch for ch in str(text).lower().strip() if ch.isalpha())
    return re.sub(r"(.)\1{2,}", r"\1\1", t)


def load_slang_lexicon(csv_path: str):
    if not csv_path or not os.path.exists(csv_path):
        print(f'[SLANG] File tidak ditemukan: {csv_path}.')
        return {}, set(), set()

    slang_to_formal = {}
    slang_terms = set()
    formal_terms = set()
    with open(csv_path, 'r', encoding='utf-8-sig', newline='') as f:
        reader = csv.DictReader(f)
        if not reader.fieldnames:
            print(f'[SLANG] Header CSV tidak ditemukan: {csv_path}.')
            return {}, set(), set()
        if 'slang' not in reader.fieldnames or 'formal' not in reader.fieldnames:
            print(f"[SLANG] Header wajib 'slang,formal' tidak lengkap: {reader.fieldnames}")
            return {}, set(), set()

        for row in reader:
            slang = _norm_lexicon_term(row.get('slang', ''))
            formal = _norm_formal_term(row.get('formal', ''))
            if not slang or not formal:
                continue
            slang_to_formal[slang] = formal
            slang_terms.add(slang)
            formal_terms.add(formal)

    print(f'[SLANG] Loaded {len(slang_to_formal)} mapping term dari: {csv_path}')
    return slang_to_formal, slang_terms, formal_terms


def build_symspell_from_terms(terms: set[str], max_edit_distance: int = 2):
    if not terms:
        print('[SymSpell] Term lexicon kosong. Recover typo dinonaktifkan.')
        return None

    symspell = SymSpell(max_dictionary_edit_distance=max_edit_distance, prefix_length=7)
    loaded = 0
    for term in sorted(terms):
        if not term or not term.isalpha():
            continue
        symspell.create_dictionary_entry(term, 1)
        loaded += 1

    if loaded == 0:
        print('[SymSpell] Semua term lexicon tidak valid. Recover typo dinonaktifkan.')
        return None

    print(f'[SymSpell] Loaded {loaded} term dari formal slang.csv')
    return symspell


slang_formal_map, slang_terms, formal_terms = load_slang_lexicon(_resolve_slang_csv_path())

symspell_model = build_symspell_from_terms(formal_terms)

In [ ]:
def _split_reduplication_token(token: str) -> list[str]:
    t = str(token or '').strip()
    if len(t) < 6 or not t.isalpha():
        return [t] if t else []

    n = len(t)
    # Pola 1: X + X (+suffix) -> [X, X+suffix]
    for stem_len in range(3, (n // 2) + 1):
        stem = t[:stem_len]
        if t[stem_len:stem_len * 2] != stem:
            continue
        suffix = t[stem_len * 2:]
        second = stem + suffix
        if second:
            return [stem, second]

    # Pola 2: prefix + X + X (+suffix) -> [prefix+X, X+suffix]
    max_prefix = min(4, n - 6)
    for prefix_len in range(2, max_prefix + 1):
        remain = t[prefix_len:]
        m = len(remain)
        for stem_len in range(3, (m // 2) + 1):
            stem = remain[:stem_len]
            if remain[stem_len:stem_len * 2] != stem:
                continue
            suffix = remain[stem_len * 2:]
            first = t[:prefix_len + stem_len]
            second = stem + suffix
            if first and second:
                return [first, second]
    return [t]


import html


def clean_raw_pantun_text(text: str) -> str:
    # Decode HTML entities sebelum normalisasi lanjutan.
    cleaned = html.unescape(str(text or ''))
    # Ubah simbol '&' menjadi kata 'dan' agar terbaca sebagai 1 silabel.
    cleaned = re.sub(r'\s*&\s*', ' dan ', cleaned)
    return cleaned.strip()


def _clean_input_words(text: str) -> list[str]:
    words = []
    raw_text = clean_raw_pantun_text(text)
    for raw in raw_text.split():
        cleaned = re.sub(r"^[^\w]+|[^\w]+$", "", raw)
        if not cleaned:
            continue
        parts = re.split(r"[-‐‑‒–—]+", cleaned)
        for part in parts:
            piece = re.sub(r"^[^\w]+|[^\w]+$", "", part)
            if not piece:
                continue
            for sub in _split_reduplication_token(piece):
                if sub:
                    words.append(sub)
    return words


def predict_token_labels(text: str):
    """Inferensi token-level label + confidence dengan penyelarasan wordpiece -> token asli."""
    words = _clean_input_words(text)
    if not words:
        return []
    enc = tokenizer(
        words,
        is_split_into_words=True,
        return_tensors='pt',
        truncation=True,
        padding=False
    )
    enc = {k: v.to(device) for k, v in enc.items()}

    with torch.no_grad():
        logits = model(**enc).logits
        probs = torch.softmax(logits, dim=-1)[0]
        pred_ids = torch.argmax(logits, dim=-1)[0].tolist()

    word_ids = tokenizer(words, is_split_into_words=True).word_ids()

    token_rows = []
    current_word = None
    for idx, widx in enumerate(word_ids):
        if widx is None:
            continue
        if widx != current_word:
            current_word = widx
            label = id2label[pred_ids[idx]]

            top2 = torch.topk(probs[idx], k=min(2, probs[idx].shape[-1]))
            confidence = float(top2.values[0].item())
            second_best = float(top2.values[1].item()) if top2.values.shape[0] > 1 else 0.0
            margin = confidence - second_best

            token_rows.append(
                {
                    'token': words[widx],
                    'label': label,
                    'confidence': confidence,
                    'margin': margin,
                }
            )
    return token_rows


LABELS_ASING = {"B-ASG", "I-ASG"}
LABELS_SINGKATAN = {"B-SKT", "I-SKT"}
LABELS_DISEMVOWEL = {"B-DSM", "I-DSM"}
LABELS_TYPO = {"B-TYP", "I-TYP"}
LABELS_VALID = {"O", *LABELS_ASING, *LABELS_SINGKATAN, *LABELS_DISEMVOWEL, *LABELS_TYPO}


def route_by_label(label: str) -> str:
    """Routing sesuai dokumentasi pipeline dan skema label BIO IndoBERT."""
    norm = label.upper().strip()
    if norm not in LABELS_VALID:
        raise ValueError(f'Label tidak dikenali: {label}')

    if norm in LABELS_SINGKATAN:
        return 'RUMUS_A_FONEMISASI_ALFABETIS'
    if norm in LABELS_DISEMVOWEL or norm in LABELS_TYPO:
        return 'SLANG_LEXICON_OR_DIRECT_G2P_ID'
    if norm in LABELS_ASING:
        return 'FAILSAFE_G2P_ASING'
    return 'CRF1_G2P_KATA_BAKU'


def _replace_label_entity(label: str, target_entity: str) -> str:
    norm = label.upper().strip()
    if norm.startswith('B-'):
        return f'B-{target_entity}'
    if norm.startswith('I-'):
        return f'I-{target_entity}'
    return target_entity


def is_likely_disemvowelling(token: str) -> bool:
    """DSM sederhana: token dianggap DSM bila ada di slang.csv."""
    mapping = globals().get('slang_formal_map') or {}
    normalized = _norm_lexicon_term(token)
    return bool(normalized and normalized in mapping)


def override_typo_to_dsm(token: str, label: str) -> str:
    norm = label.upper().strip()
    if norm in LABELS_DISEMVOWEL or norm in LABELS_TYPO:
        if is_likely_disemvowelling(token):
            return _replace_label_entity(norm, 'DSM')
        return _replace_label_entity(norm, 'TYP')
    return norm


def _is_stem_in_sastrawi_root_words(token: str) -> bool:
    if not isinstance(token, str):
        return False

    t = normalize_typo_token(token)
    if len(t) < 2 or not t.isalpha():
        return False

    stemmer = globals().get('_id_stemmer')
    root_words = globals().get('ID_SASTRAWI_ROOT_WORDS')
    if stemmer is None or root_words is None:
        from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
        factory = StemmerFactory()
        stemmer = factory.create_stemmer()
        root_words = set(factory.get_words())
        globals()['_id_stemmer'] = stemmer
        globals()['ID_SASTRAWI_ROOT_WORDS'] = root_words

    lemma = stemmer.stem(t)
    return bool(lemma) and lemma in root_words


# Kluster onset & koda sah untuk membedakan akronim silabik vs singkatan eja huruf.
_VALID_ID_ONSETS = {
    '', 'b', 'c', 'd', 'f', 'g', 'h', 'j', 'k', 'l', 'm', 'n', 'p', 'q', 'r', 's', 't', 'v', 'w', 'y', 'z',
    'ng', 'ny', 'sy', 'kh', 'pr', 'pl', 'br', 'bl', 'tr', 'dr', 'kr', 'kl', 'gr', 'gl', 'fr', 'fl', 'sr', 'sl', 'sw', 'st', 'sp', 'sk'
}
_VALID_ID_CODAS = {'', 'b', 'd', 'f', 'g', 'h', 'k', 'l', 'm', 'n', 'p', 'r', 's', 't', 'w', 'y', 'z', 'ng', 'ny', 'kh', 'sy'}


def _is_pronounceable_id_syllabic_word(token: str) -> bool:
    t = re.sub(r'[^a-z]', '', str(token or '').lower())
    if len(t) < 3:
        return False

    stemmer = globals().get('_id_stemmer')
    root_words = globals().get('ID_SASTRAWI_ROOT_WORDS')
    if stemmer is None or root_words is None:
        from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
        factory = StemmerFactory()
        stemmer = factory.create_stemmer()
        root_words = set(factory.get_words())
        globals()['_id_stemmer'] = stemmer
        globals()['ID_SASTRAWI_ROOT_WORDS'] = root_words

    lemma = stemmer.stem(t)
    if lemma in root_words:
        return True

    if not re.search(r'[aiueo]', t):
        return False

    parts = re.split(r'[aiueo]+', t)
    if parts[0] not in _VALID_ID_ONSETS or parts[-1] not in _VALID_ID_CODAS:
        return False

    for medial in parts[1:-1]:
        valid_split = any(
            medial[:i] in _VALID_ID_CODAS and medial[i:] in _VALID_ID_ONSETS
            for i in range(len(medial) + 1)
        )
        if not valid_split:
            return False

    return True


def sanitize_switchboard_tag(token: str, predicted_tag: str) -> str:
    t = normalize_typo_token(token)
    norm = str(predicted_tag or '').upper().strip()
    if not t:
        return norm

    if norm in LABELS_SINGKATAN and _is_pronounceable_id_syllabic_word(t):
        return 'O'

    return norm


def override_low_conf_asg_to_o(label: str, token_meta: dict | None = None) -> str:
    norm = label.upper().strip()
    if norm not in LABELS_ASING:
        return norm

    token = token_meta.get('token') if isinstance(token_meta, dict) else None
    if isinstance(token, str) and token.lower().strip() in ID_BOUND_MORPHEMES:
        return 'O'

    if not isinstance(token_meta, dict):
        return norm

    if _is_stem_in_sastrawi_root_words(token):
        return 'O'

    return norm


from functools import lru_cache



def is_likely_foreign_token(token: str) -> bool:
    t = normalize_typo_token(token)
    if len(t) < 3 or not t.isalpha():
        return False

    en_phon = g2p_foreign_phoneme(t)
    if not en_phon:
        return False
    id_phon = g2p_id_phoneme(t)

    foreign_markers = ('æ', 'ɝ', 'ʌ', 'θ', 'ð', 'ʃ', 'ʒ', 'tʃ', 'dʒ', 'eɪ', 'aɪ', 'oʊ', 'ɔɪ', 'aʊ')
    if any(marker in en_phon for marker in foreign_markers):
        return True

    if id_phon and en_phon != id_phon:
        return True
    return False


@lru_cache(maxsize=8192)
def is_in_english_vocabulary(token: str) -> bool:
    t = normalize_typo_token(token)
    if len(t) < 2 or not t.isalpha():
        return False

    words = globals().get('_cmudict_words_set')
    if words is None:
        try:
            from nltk.corpus import cmudict
            words = set(cmudict.words())
        except LookupError:
            print('[CMUDICT] Resource belum tersedia. Cek _ensure_nltk_resource(corpora/cmudict).')
            return False
        globals()['_cmudict_words_set'] = words

    return t in words

ID_BOUND_MORPHEMES = {
    'nya', 'ku', 'mu', 'di', 'ke', 'se',
    'kah', 'lah', 'pun', 'nge', 'meng', 'ter', 'ber'
}

def override_low_conf_o_to_asg(token: str, label: str, token_meta: dict | None = None, raw_label: str | None = None) -> str:
    norm = label.upper().strip()
    raw_norm = str(raw_label or '').upper().strip()
    if norm != 'O' or raw_norm != 'O':
        return norm

    if token.lower().strip() in ID_BOUND_MORPHEMES:
        return 'O'

    if not isinstance(token_meta, dict):
        return norm

    conf = token_meta.get('confidence')
    margin = token_meta.get('margin')

    # Turunkan default dari (0.78, 0.50) menjadi (0.65, 0.20)
    conf_th = float(globals().get('O_TO_ASG_CONF_THRESHOLD', 0.65))
    margin_th = float(globals().get('O_TO_ASG_MARGIN_THRESHOLD', 0.20))

    # Gunakan 'and' agar kata baku Indonesia yang cukup yakin tidak ikut terubah
    is_low_conf = conf is not None and conf < conf_th
    is_low_margin = margin is not None and margin < margin_th

    if is_low_conf and is_low_margin:
        if is_in_english_vocabulary(token):
            return 'B-ASG'
        return 'O'

    return norm



def predict_with_routing(text: str):
    labeled = predict_token_labels(text)
    rows = []
    for item in labeled:
        tok = item['token']
        lab = item['label']
        token_meta = {
            'token': tok,
            'confidence': item.get('confidence'),
            'margin': item.get('margin'),
        }

        final_label = override_typo_to_dsm(tok, lab)
        final_label = override_low_conf_asg_to_o(final_label, token_meta)
        final_label = override_low_conf_o_to_asg(tok, final_label, token_meta, raw_label=lab)
        final_label = sanitize_switchboard_tag(tok, final_label)
        switch = acoustic_switchboard(tok, final_label, token_meta=token_meta)
        units_now = phoneme_to_units(switch['phoneme'])

        bi_pattern = None
        bi_source = None
        if not str(switch.get('switch_route', '')).startswith('ASG_'):
            bi_fn = globals().get('predict_bi_from_phoneme')
            bi_pattern = bi_fn(switch['phoneme'], token=tok) if callable(bi_fn) else None
            bi_source = 'CRF3_BI' if bi_pattern else None

        if str(switch.get('switch_route', '')).startswith('ASG_'):
            syll_fn = globals().get('syllabify_en_ipa_robust')
            asg_syll = syll_fn(units_now) if callable(syll_fn) else None
            if isinstance(asg_syll, dict):
                bi_pattern = asg_syll.get('bi') or ''
            else:
                bi_pattern = ''
            bi_source = 'ASG_FONOTAKTIK_BI' if bi_pattern else None

        route_now = str(switch.get('switch_route', ''))
        use_id_orth_patch = bool(globals().get('ENABLE_ID_ORTHOGRAPHIC_BI_PATCH', False))
        if use_id_orth_patch and 'G2P_ID' in route_now:
            refine_id_fn = globals().get('refine_id_bi_with_orthography')
            if callable(refine_id_fn):
                refined_bi, refined_source = refine_id_fn(tok, units_now, bi_pattern, route_now)
                if refined_bi:
                    bi_pattern = refined_bi
                    bi_source = refined_source or bi_source

        rows.append(
            {
                'token': tok,
                'raw_label': lab,
                'label': final_label,
                'route': route_by_label(final_label),
                'model_confidence': token_meta['confidence'],
                'model_margin': token_meta['margin'],
                'switch_route': switch['switch_route'],
                'switch_input': switch['switch_input'],
                'phoneme': switch['phoneme'],
                'bi_pattern': bi_pattern,
                'bi_source': bi_source,
                'bc_info': switch.get('bc_info')
            }
        )
    return rows


In [ ]:
# Helper sederhana Rumus A (singkatan)
from functools import lru_cache
import os
import re
import nltk
import joblib
from typing import Any
from symspellpy import Verbosity
from g2p_id import G2P as G2P_ID
from g2p_en import G2p as G2P_EN


def _ensure_nltk_resource(resource_path: str, package_name: str):
    try:
        nltk.data.find(resource_path)
    except LookupError:
        nltk.download(package_name, quiet=True)


_ensure_nltk_resource('corpora/cmudict', 'cmudict')
_ensure_nltk_resource('taggers/averaged_perceptron_tagger_eng', 'averaged_perceptron_tagger_eng')
_ensure_nltk_resource('taggers/averaged_perceptron_tagger', 'averaged_perceptron_tagger')

g2p_id_model = G2P_ID()
g2p_en_model = G2P_EN()

ALPHABET_EJA_MAP = {
    "a": "a", "b": "be", "c": "ce", "d": "de", "e": "e", "f": "ef", "g": "ge",
    "h": "ha", "i": "i", "j": "je", "k": "ka", "l": "el", "m": "em", "n": "en",
    "o": "o", "p": "pe", "q": "ki", "r": "er", "s": "es", "t": "te", "u": "u",
    "v": "ve", "w": "we", "x": "eks", "y": "ye", "z": "zet"
}

# Konversi simbol fonem Inggris (ARPAbet) ke fonetik internasional (IPA)
ARPABET_TO_IPA = {
    "AA": "ɑ", "AE": "æ", "AH": "ʌ", "AO": "ɔ", "AW": "aʊ", "AY": "aɪ",
    "B": "b", "CH": "tʃ", "D": "d", "DH": "ð", "EH": "ɛ", "ER": "ɝ",
    "EY": "eɪ", "F": "f", "G": "ɡ", "HH": "h", "IH": "ɪ", "IY": "i",
    "JH": "dʒ", "K": "k", "L": "l", "M": "m", "NG": "ŋ", "OW": "oʊ",
    "OY": "ɔɪ", "P": "p", "R": "r", "S": "s", "SH": "ʃ", "T": "t",
    "TH": "θ", "UH": "ʊ", "UW": "u", "V": "v", "W": "w", "Y": "j",
    "Z": "z", "ZH": "ʒ"
}




CRF3_VOWELS = {"a", "i", "u", "e", "o", "ə", "ɛ", "ɪ", "ʊ", "ɔ", "æ", "ɐ", "ɒ", "ɑ", "ʌ", "ɝ"}
crf_bi_model = globals().get('crf_bi_model')


def load_crf_bi_model(model_path=None):
    """Load model CRF layer-3 (.joblib)."""
    global crf_bi_model
    target = str(model_path or globals().get('CRF_BI_MODEL_PATH', '')).strip()
    if not target:
        print('[CRF3] Path model kosong. Prediksi BI dinonaktifkan.')
        crf_bi_model = None
        return None
    if not os.path.exists(target):
        print(f'[CRF3] Model tidak ditemukan: {target}. Prediksi BI dinonaktifkan.')
        crf_bi_model = None
        return None

    crf_bi_model = joblib.load(target)
    print(f'[CRF3] Loaded model: {target}')
    return crf_bi_model


CRF3_DIPHTHONGS = ("eɪ", "aɪ", "aʊ", "ɔɪ", "oʊ")

IPA_VOWELS = {
    'i', 'ɪ', 'e', 'ɛ', 'æ', 'ɑ', 'ɒ', 'ɔ', 'o', 'ʊ', 'u', 'ʌ', 'ə', 'ɜ',
    'ɝ', 'ɚ', 'eɪ', 'aɪ', 'ɔɪ', 'aʊ', 'oʊ', 'əʊ', 'ɪə', 'eə', 'ʊə', 'ju',
    'iː', 'ɑː', 'ɔː', 'uː', 'ɜː'
}

SPLIT_DIPHTHONGS = {
    ('e', 'ɪ'), ('a', 'ɪ'), ('ɔ', 'ɪ'), ('a', 'ʊ'),
    ('o', 'ʊ'), ('ə', 'ʊ'), ('ɪ', 'ə'), ('e', 'ə'), ('ʊ', 'ə')
}

VALID_ONSETS = {
    'p', 'b', 't', 'd', 'k', 'g', 'f', 'v', 'θ', 'ð', 's', 'z', 'ʃ', 'ʒ', 'h',
    'm', 'n', 'ɲ', 'l', 'r', 'j', 'w', 'tʃ', 'dʒ',
    'pl', 'pr', 'bl', 'br', 'tr', 'dr', 'kl', 'kr', 'gl', 'gr',
    'fl', 'fr', 'θr', 'ʃr', 'sl', 'sm', 'sn', 'sp', 'st', 'sk', 'sw',
    'tw', 'dw', 'kw', 'gw', 'θw', 'pj', 'bj', 'tj', 'dj', 'kj', 'gj',
    'fj', 'vj', 'θj', 'sj', 'zj', 'hj', 'mj', 'nj', 'lj',
    'spl', 'spr', 'str', 'skr', 'skw', 'spj', 'stj', 'skj'
}

SYLLABIC_CONSONANTS = {'l', 'n', 'm', 'ŋ', 'r'}
OBSTRUENTS = {'p', 'b', 't', 'd', 'k', 'g', 'f', 'v', 'θ', 'ð', 's', 'z', 'ʃ', 'ʒ', 'tʃ', 'dʒ'}


def phoneme_to_units(phoneme: str) -> list[str]:
    text = str(phoneme).strip()
    if not text:
        return []
    if ' ' in text:
        return [u for u in text.split() if u]

    chars = [ch for ch in text if not ch.isspace()]
    units: list[str] = []
    i = 0
    while i < len(chars):
        if i + 1 < len(chars):
            pair = chars[i] + chars[i + 1]
            if pair in CRF3_DIPHTHONGS:
                units.append(pair)
                i += 2
                continue
        units.append(chars[i])
        i += 1
    return units


def _normalize_ipa_units(raw_units: list[str]) -> list[str]:
    cleaned = [u.replace('ˈ', '').replace('ˌ', '') for u in raw_units if u not in {'ˈ', 'ˌ', ''}]
    merged = []
    i = 0
    while i < len(cleaned):
        if i + 1 < len(cleaned) and (cleaned[i], cleaned[i + 1]) in SPLIT_DIPHTHONGS:
            merged.append(cleaned[i] + cleaned[i + 1])
            i += 2
        else:
            merged.append(cleaned[i])
            i += 1
    return merged


def _find_nuclei_indices(units: list[str]) -> list[int]:
    nuclei = []
    n = len(units)
    for i, u in enumerate(units):
        if u in IPA_VOWELS or u.endswith('̩'):
            nuclei.append(i)
        elif i == n - 1 and u in SYLLABIC_CONSONANTS and i > 0:
            if units[i - 1] in OBSTRUENTS and (not nuclei or nuclei[-1] < i - 1):
                nuclei.append(i)
    return nuclei


def syllabify_en_ipa_robust(raw_units: list[str]) -> dict[str, Any]:
    if not raw_units:
        return {'syllables': [], 'syllable_count': 0, 'bi': '', 'bi_valid': False}

    units = _normalize_ipa_units(raw_units)
    nuclei = _find_nuclei_indices(units)

    if len(nuclei) <= 1:
        bi_str = 'B' + 'I' * (len(units) - 1)
        return {
            'syllables': [''.join(units)],
            'syllable_count': 1,
            'bi': bi_str,
            'bi_valid': True,
        }

    split_points = [0]
    for idx in range(len(nuclei) - 1):
        v1 = nuclei[idx]
        v2 = nuclei[idx + 1]
        consonants = units[v1 + 1 : v2]
        num_c = len(consonants)

        if num_c == 0:
            split_points.append(v2)
        elif num_c == 1:
            if consonants[0] == 'ŋ':
                split_points.append(v2)
            else:
                split_points.append(v1 + 1)
        else:
            best_split = v2
            for c_idx in range(num_c):
                candidate_onset = ''.join(consonants[c_idx:])
                if candidate_onset in VALID_ONSETS:
                    best_split = v1 + 1 + c_idx
                    break
            split_points.append(best_split)

    split_points.append(len(units))

    syllables = []
    bi_tags = []
    for i in range(len(split_points) - 1):
        start = split_points[i]
        end = split_points[i + 1]
        chunk = units[start:end]
        if chunk:
            syllables.append(''.join(chunk))
            bi_tags.append('B' + 'I' * (len(chunk) - 1))

    bi_final = ''.join(bi_tags)
    bi_valid = (len(bi_final) == len(units)) and bi_final.startswith('B')
    return {
        'syllables': syllables,
        'syllable_count': len(syllables),
        'bi': bi_final,
        'bi_valid': bi_valid,
    }


def _crf3_is_vowel(phone: str) -> bool:
    return phone in CRF3_VOWELS


def _crf3_phone_features(seq: list[str], i: int) -> dict:
    cur = seq[i]
    prev = seq[i - 1] if i > 0 else '<BOS>'
    nxt = seq[i + 1] if i < len(seq) - 1 else '<EOS>'
    return {
        'bias': 1.0,
        'phone': cur,
        'prev_phone': prev,
        'next_phone': nxt,
        'bigram_prev_cur': f'{prev}+{cur}',
        'bigram_cur_next': f'{cur}+{nxt}',
        'is_vowel': _crf3_is_vowel(cur),
        'prev_is_vowel': False if prev == '<BOS>' else _crf3_is_vowel(prev),
        'next_is_vowel': False if nxt == '<EOS>' else _crf3_is_vowel(nxt),
        'pos': i,
        'rev_pos': len(seq) - 1 - i,
        'len_seq': len(seq),
        'bos': i == 0,
        'eos': i == len(seq) - 1,
    }

from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

_id_stemmer_factory = StemmerFactory()
_id_stemmer = _id_stemmer_factory.create_stemmer()
ID_SASTRAWI_ROOT_WORDS = set(_id_stemmer_factory.get_words())
ID_DIPHTHONG_PAIRS = {('a', 'i'), ('a', 'u'), ('o', 'i'), ('e', 'i'), ('ɛ', 'i')}
ID_TRUE_MONOSYLLABLE_DIPHTHONGS = {'kau', 'hai', 'hay', 'hei', 'hey', 'boi', 'boy'}
ID_TRUE_HIATUS_DIPHTHONG_LEMMAS = {'mau', 'bau', 'tau', 'dai', 'kue', 'goa', 'ria', 'dia', 'ia'}
def _is_suffixed_ai_kbbi(token: str) -> bool:
    """Deteksi sufiks -i secara morfologis via Sastrawi tanpa daftar kata hardcoded."""
    t = normalize_typo_token(token)
    if not t.endswith('ai') or len(t) < 5:
        return False

    lemma = _id_stemmer.stem(t)
    # Jika bentuk dasarnya berakhiran '-a' (bukan '-ai'), berarti akhiran '-i' adalah imbuhan
    return lemma.endswith('a') and not lemma.endswith('ai')


# Kluster onset 2-konsonan yang sah dalam fonotaktik bahasa Indonesia
ID_VALID_ONSET_CLUSTERS = {
    'pr', 'pl', 'br', 'bl', 'tr', 'dr', 'kr', 'kl', 'gr', 'gl',
    'fr', 'fl', 'sr', 'sl', 'sw', 'kw', 'sp', 'st', 'sk',
    'dʒ', 'tʃ'  # afrikat tunggal yang direpresentasikan 2 karakter oleh g2p-id
}

def repair_id_consonant_clusters_and_nuclei(units: list[str], bi_pattern: str) -> str:
    """
    Memperbaiki prediksi BI CRF-3 yang menghasilkan suku kata tanpa vokal
    (mis. ['nɔ', 'ŋk', 'rɔŋ'] -> ['nɔŋ', 'krɔŋ'], ['dʒɔ', 'mb', 'lo'] -> ['dʒɔm', 'blo']).
    """
    if not units or not bi_pattern or len(units) != len(bi_pattern):
        return bi_pattern

    n = len(units)
    is_v = [_is_vowel_unit_for_nucleus(u) for u in units]
    vowel_indices = [i for i, v in enumerate(is_v) if v]

    # Jika tidak ada vokal sama sekali, jadikan 1 silabel
    if not vowel_indices:
        return 'B' + ('I' * (n - 1))

    # Cek apakah hasil BI saat ini memiliki suku kata tanpa vokal atau jumlah B != jumlah inti vokal
    # (kecuali deret vokal berurutan yang memang terpisah)
    tags = list(bi_pattern)
    tags[0] = 'B'

    # Deteksi apakah ada segmen B...I yang tidak mengandung vokal sama sekali
    has_vowelless_syllable = False
    cur_has_vowel = is_v[0]
    for i in range(1, n):
        if tags[i] == 'B':
            if not cur_has_vowel:
                has_vowelless_syllable = True
                break
            cur_has_vowel = is_v[i]
        else:
            cur_has_vowel = cur_has_vowel or is_v[i]
    if not cur_has_vowel:
        has_vowelless_syllable = True

    if not has_vowelless_syllable:
        return ''.join(tags)

    # Rekonstruksi batas silabel berbasis inti vokal + Maximum Onset Principle
    rebuilt = ['I'] * n
    rebuilt[0] = 'B'

    for k in range(len(vowel_indices) - 1):
        v1 = vowel_indices[k]
        v2 = vowel_indices[k + 1]
        num_c = v2 - v1 - 1

        if num_c == 0:
            # Dua vokal berurutan (hiatus): V2 memulai silabel baru
            rebuilt[v2] = 'B'
        elif num_c == 1:
            # V-C-V -> V-CV (konsonan tunggal jadi onset silabel berikutnya)
            rebuilt[v1 + 1] = 'B'
        else:
            # V-C1...Cm-V -> cari onset kluster terpanjang yang sah di depan V2
            split_idx = v2 - 1  # default: hanya 1 konsonan terakhir sebelum V2 yang jadi onset
            if num_c >= 2:
                pair = units[v2 - 2] + units[v2 - 1]
                if pair in ID_VALID_ONSET_CLUSTERS:
                    split_idx = v2 - 2
            rebuilt[split_idx] = 'B'

    return ''.join(rebuilt)


# Pasangan vokal yang boleh berbagi 1 silabel (selain ini wajib dipisah sebagai hiatus)
_ALLOWED_DIPHTHONG_NUCLEI = {
    ('a', 'i'), ('a', 'u'), ('o', 'i'),
    ('ɑ', 'i'), ('ɑ', 'u'), ('ɔ', 'i'),
}

def split_non_diphthong_hiatus(units: list[str], bi_pattern: str) -> str:
    """
    Memisahkan deret vokal non-diftong (hiatus) seperti:
    - Batas prefiks: se-u-tuh-nya, di-o-mo-ngin, di-u-jung, se-u-mur, se-e-kor
    - Hiatus leksikal: ku-e, ra-di-o-nya, ma-in, sa-at
    """
    if not units or not bi_pattern or len(units) != len(bi_pattern):
        return bi_pattern

    tags = list(bi_pattern)
    for i in range(1, len(units)):
        u_prev = units[i - 1].lower()
        u_curr = units[i].lower()
        if _is_vowel_unit_for_nucleus(u_prev) and _is_vowel_unit_for_nucleus(u_curr):
            if (u_prev, u_curr) not in _ALLOWED_DIPHTHONG_NUCLEI:
                tags[i] = 'B'
    return ''.join(tags)

def _has_inflectional_i_suffix(token: str) -> bool:
    """
    Deteksi apakah akhiran '-i' pada token berakhiran '-ai' adalah sufiks '-i'
    (cintai -> cinta, menghargai -> harga) atau diftong leksikal kata dasar
    (beramai -> ramai, pantai -> pantai).
    """
    t = normalize_typo_token(token)
    if not t.endswith('ai') or len(t) < 5:
        return False

    # Kaidah morfologi BI: prefiks ber-, ter-, se-, ke- tidak pernah berpasangan dengan sufiks -i
    if t.startswith(('ber', 'be', 'ter', 'te', 'se', 'ke')):
        return False

    full_lemma = _id_stemmer.stem(t)
    if full_lemma.endswith('ai') or (full_lemma + 'i') in ID_SASTRAWI_ROOT_WORDS:
        return False

    return (
        full_lemma.endswith('a')
        and not full_lemma.endswith('ai')
        and full_lemma in ID_SASTRAWI_ROOT_WORDS
    )


def postprocess_id_diphthong_bi(units: list[str], bi_pattern: str, token: str = '') -> str:
    """Penyelarasan diftong terbuka Indonesia (ai, au, oi, ei) sesuai kaidah KBBI."""
    if not units or not bi_pattern or len(units) != len(bi_pattern):
        return bi_pattern

    t = normalize_typo_token(token)
    tags = list(bi_pattern)
    n = len(units)

    # 1. Kata pendek 3 fonem K-V-V (kau/hai/hei -> 1 silabel; mau/bau/dai -> 2 silabel)
    if n == 3 and (units[-2], units[-1]) in ID_DIPHTHONG_PAIRS:
        tags[-1] = 'I' if t in ID_TRUE_MONOSYLLABLE_DIPHTHONGS else 'B'
        return ''.join(tags)

    lemma = _id_stemmer.stem(t) if t else ''
    if lemma == t and t.endswith('in') and len(t) >= 5:
        cand = _id_stemmer.stem(t[:-2])
        if cand in ID_SASTRAWI_ROOT_WORDS:
            lemma = cand

    # 1b. Hiatus leksikal berbasis lemma (tau, mau, bau, dai)
    if lemma in ID_TRUE_HIATUS_DIPHTHONG_LEMMAS:
        for i in range(1, n):
            if (units[i - 1], units[i]) in ID_DIPHTHONG_PAIRS:
                tags[i] = 'B'
        return ''.join(tags)

    # 2. Hiatus tertutup pada kata dasar (jauh -> berjauhan, laut -> lautan, baik -> kebaikan) -> PISAH (B)
    if lemma and re.search(r'(ai|au|oi)[bcdfghjklmnpqrstvwxyz]$', lemma):
        for i in range(1, n):
            if (units[i - 1], units[i]) in ID_DIPHTHONG_PAIRS:
                tags[i] = 'B'
        return ''.join(tags)

    # 3. Sufiks -i pada kata dasar berakhiran -a (cin-ta-i, meng-har-ga-i) -> PISAH (B)
    if _has_inflectional_i_suffix(t):
        if tags[-1] == 'I':
            tags[-1] = 'B'
        return ''.join(tags)

    # 4. Diftong leksikal akhir kata terbuka -> GABUNG (I)
    lemma_has_open_diphthong_tail = bool(lemma and re.search(r'(ai|au|oi)$', lemma))
    if (n >= 4 or lemma_has_open_diphthong_tail) and (units[-2], units[-1]) in ID_DIPHTHONG_PAIRS:
        if tags[-1] == 'B':
            tags[-1] = 'I'

    # 5. Diftong leksikal di tengah kata/sebelum sufiks terbuka (permaisuri, saudara, pantainya) -> GABUNG (I)
    if lemma and re.search(r'(ai|au|oi)(?:$|[bcdfghjklmnpqrstvwxyz][aiueo])', lemma):
        for i in range(1, n):
            if (units[i - 1], units[i]) in ID_DIPHTHONG_PAIRS:
                tags[i] = 'I'

    return ''.join(tags)


def predict_bi_from_phoneme(phoneme: str, model=None, token: str = ''):
    mdl = model if model is not None else globals().get('crf_bi_model')
    if mdl is None:
        return None

    units = phoneme_to_units(phoneme)
    if not units:
        return None

    feats = [_crf3_phone_features(units, i) for i in range(len(units))]
    pred = mdl.predict_single(feats)
    raw_bi = ''.join(pred)

    # 1. Pastikan setiap silabel punya vokal & kluster konsonan sah (nongkrong, jomblo)
    clean_bi = repair_id_consonant_clusters_and_nuclei(units, raw_bi)
    # 2. Pisahkan semua deret vokal non-diftong / hiatus (se-u-tuh-nya, di-o-mo-ngin, ku-e, ra-di-o-nya)
    clean_bi = split_non_diphthong_hiatus(units, clean_bi)
    # 3. Selaraskan diftong vs hiatus pada ai/au/oi sesuai kaidah KBBI & Sastrawi
    return postprocess_id_diphthong_bi(units, clean_bi, token=token)

def _count_b_from_bi(bi_pattern: str | None) -> int:
    return sum(1 for ch in (bi_pattern or '') if ch == 'B')


def _orthographic_vowel_group_count(token: str) -> int:
    t = normalize_typo_token(token)
    if not t:
        return 0
    return len(re.findall(r'[aiueo]+', t))


def build_bi_from_target_syllables(units: list[str], target_syllables: int) -> str | None:
    if not units:
        return None

    n = len(units)
    target = max(1, min(int(target_syllables), n))
    if target == 1:
        return 'B' + ('I' * (n - 1))

    labels = ['I'] * n
    labels[0] = 'B'

    candidate = [i for i in range(1, n) if _is_vowel_unit_for_nucleus(units[i])]
    if len(candidate) < target - 1:
        candidate = sorted(set(candidate + list(range(1, n))))

    if not candidate:
        return 'B' + ('I' * (n - 1))

    k = target - 1
    if k >= len(candidate):
        chosen = candidate
    else:
        chosen = []
        for j in range(k):
            idx = round(j * (len(candidate) - 1) / (k - 1)) if k > 1 else 0
            chosen.append(candidate[idx])
        chosen = sorted(set(chosen))

    for idx in chosen:
        labels[idx] = 'B'

    return ''.join(labels)


def build_id_bi_from_target_syllables(units: list[str], target_syllables: int) -> str | None:
    if not units:
        return None

    n = len(units)
    target = max(1, min(int(target_syllables), n))
    if target == 1:
        return 'B' + ('I' * (n - 1))

    first_vowel_idx = next((i for i, u in enumerate(units) if _is_vowel_unit_for_nucleus(u)), None)
    min_start = (first_vowel_idx + 1) if first_vowel_idx is not None else 1

    starts = []
    for i in range(1, n):
        if i < min_start:
            continue
        prev_u = units[i - 1]
        cur_u = units[i]
        prev_is_vowel = _is_vowel_unit_for_nucleus(prev_u)
        cur_is_vowel = _is_vowel_unit_for_nucleus(cur_u)
        if ((not prev_is_vowel) and cur_is_vowel) or (prev_is_vowel and cur_is_vowel):
            starts.append(i)

    if len(starts) < target - 1:
        starts = sorted(set(starts + list(range(min_start, n))))
    if not starts:
        return build_bi_from_target_syllables(units, target)

    k = target - 1
    if k >= len(starts):
        chosen = starts
    else:
        chosen = []
        for j in range(k):
            idx = round(j * (len(starts) - 1) / (k - 1)) if k > 1 else 0
            chosen.append(starts[idx])
        chosen = sorted(set(chosen))

    labels = ['I'] * n
    labels[0] = 'B'
    for idx in chosen:
        labels[idx] = 'B'
    return ''.join(labels)


def refine_id_bi_with_orthography(
    token: str,
    units: list[str],
    bi_pattern: str | None,
    switch_route: str | None = None,
) -> tuple[str | None, str | None]:
    if not units:
        return bi_pattern, None
    route = str(switch_route or '')
    if 'G2P_ID' not in route:
        return bi_pattern, None

    target_count = _orthographic_vowel_group_count(token)
    if target_count <= 1:
        return bi_pattern, None

    crf_count = _count_b_from_bi(bi_pattern)
    collapsed = crf_count <= 1 and len(units) >= 4
    strongly_mismatch = crf_count > 0 and abs(crf_count - target_count) >= 2
    if collapsed or strongly_mismatch:
        rebuilt = build_id_bi_from_target_syllables(units, target_count)
        if rebuilt:
            return rebuilt, 'ID_ORTHOGRAPHIC_BI'

    return bi_pattern, None


_ID_ENCLITIC_PHONEMES = {
    'nya': 'ɲa',
    'ku': 'ku',
    'mu': 'mu',
}

def _split_id_enclitic(token: str) -> tuple[str, str]:
    t = normalize_typo_token(token)
    for clitic in ('nya', 'ku', 'mu'):
        if t.endswith(clitic) and len(t) > len(clitic) + 2:
            return t[:-len(clitic)], clitic
    return t, ''

def choose_asg_phoneme(token: str) -> dict:
    cleaned = normalize_typo_token(token)
    stem, clitic = _split_id_enclitic(cleaned)

    # Jika setelah klitik dilepas akarnya ada di kamus Indonesia atau slang.csv, alihkan ke G2P_ID
    recovered = recover_formal_from_slang(stem)
    if recovered:
        base_phoneme = g2p_id_phoneme(recovered['recovered'])
        clitic_phoneme = _ID_ENCLITIC_PHONEMES.get(clitic, '')
        return {
            'switch_route': 'ASG_RESCUED_SLANG_G2P_ID',
            'switch_input': recovered['recovered'] + clitic,
            'phoneme': base_phoneme + clitic_phoneme,
        }

    if _is_stem_in_sastrawi_root_words(stem):
        base_phoneme = g2p_id_phoneme(stem)
        clitic_phoneme = _ID_ENCLITIC_PHONEMES.get(clitic, '')
        return {
            'switch_route': 'ASG_RESCUED_ROOT_G2P_ID',
            'switch_input': cleaned,
            'phoneme': base_phoneme + clitic_phoneme,
        }

    # Jika benar-benar kata asing + klitik Indonesia (mis. landingnya, photomu)
    base_phoneme = g2p_foreign_phoneme(stem)
    clitic_phoneme = _ID_ENCLITIC_PHONEMES.get(clitic, '')
    return {
        'switch_route': 'ASG_G2P_EN_IPA',
        'switch_input': cleaned,
        'phoneme': base_phoneme + clitic_phoneme,
    }


def singkatan_to_ejaan(word: str) -> str:
    """dpr -> de pe er"""
    clean = "".join(ch for ch in word.lower() if ch.isalpha())
    return " ".join(ALPHABET_EJA_MAP.get(ch, ch) for ch in clean)


def arpabet_to_ipa(arpabet_tokens: list[str]) -> str:
    ipa_tokens = []
    for token in arpabet_tokens:
        base = re.sub(r"\d", "", token).upper()
        if not base:
            continue
        ipa_tokens.append(ARPABET_TO_IPA.get(base, base.lower()))
    return "".join(ipa_tokens)


def _coerce_g2p_output_to_str(raw) -> str:
    if isinstance(raw, tuple):
        return _coerce_g2p_output_to_str(raw[0])
    if isinstance(raw, list):
        if raw and isinstance(raw[0], list):
            raw = [ph for sub in raw for ph in sub if ph]
        return " ".join(str(part) for part in raw if part)
    return str(raw).strip()


@lru_cache(maxsize=8192)
def g2p_id_phoneme(word: str) -> str:
    if not word:
        return ""
    try:
        res = g2p_id_model.to_phoneme(word)
    except AttributeError:
        try:
            res = g2p_id_model(word)
        except Exception:
            res = g2p_id_model.convert(word)
    return _coerce_g2p_output_to_str(res).replace(" ", "")


@lru_cache(maxsize=8192)
def g2p_foreign_phoneme(word: str) -> str:
    if not word:
        return ""
    raw = g2p_en_model(word)
    phones = [p for p in raw if p and p != " " and p != "<unk>"]
    ipa = arpabet_to_ipa(phones)
    return ipa if ipa else g2p_id_phoneme(word)


def rumus_a_singkatan_simple(word: str) -> str:
    """Rumus A: singkatan -> ejaan alfabet -> fonem Indonesia (g2p-id)."""
    ejaan = singkatan_to_ejaan(word)
    if not ejaan:
        return ""
    return g2p_id_phoneme(ejaan)


# -------------------------------
# Slang lexicon + typo handling (tanpa hardcoded kata)
# -------------------------------

VOWELS = set('aiueo')


def normalize_typo_token(token: str) -> str:
    """Normalisasi typo berbasis bentuk: lowercase, alfabet-only, kompres pengulangan huruf."""
    t = token.lower().strip()
    if not t:
        return t
    t = "".join(ch for ch in t if ch.isalpha())
    if not t:
        return t
    t = re.sub(r"(.)\1{2,}", r"\1\1", t)
    return t


@lru_cache(maxsize=8192)
def symspell_recover_typo(
    token: str,
    allow_no_vowel: bool = False,
    max_edit_distance: int = 2,
    max_len_delta: int = 3,
) -> dict | None:
    """Recover typo ringan; return None jika tidak yakin."""
    model = globals().get('symspell_model')
    if model is None:
        return None

    normalized = normalize_typo_token(token)
    if not normalized:
        return None
    if len(normalized) < 2:
        return None

    if (not allow_no_vowel) and all(ch not in VOWELS for ch in normalized):
        return None

    suggestions = model.lookup(
        normalized,
        Verbosity.TOP,
        max_edit_distance=max_edit_distance,
        include_unknown=True,
        transfer_casing=False,
    )
    if not suggestions:
        return None

    top = suggestions[0]
    recovered = top.term.lower().strip()
    if not recovered or recovered == normalized:
        return None

    allowed_dist = 1 if len(normalized) <= 5 else 2
    allowed_dist = min(allowed_dist, max_edit_distance)
    if top.distance > allowed_dist:
        return None
    if abs(len(recovered) - len(normalized)) > max_len_delta:
        return None

    return {
        'original': normalized,
        'recovered': recovered,
        'distance': int(top.distance),
        'count': int(top.count),
    }


def recover_formal_from_slang(token: str) -> dict | None:
    """Recover formal via map slang.csv dengan exact match setelah normalisasi."""
    mapping = globals().get('slang_formal_map') or {}
    if not mapping:
        return None

    normalized = _norm_lexicon_term(token)
    if not normalized:
        return None

    formal = mapping.get(normalized)
    if not formal:
        return None

    return {
        'original': normalized,
        'matched_slang': normalized,
        'recovered': formal,
        'source': 'exact',
    }


def _recover_formal_with_symspell(token: str) -> dict | None:
    """Recover typo ke kolom formal slang.csv via SymSpell (tanpa map balik slang)."""
    max_edit_distance = int(globals().get('SYMSPELL_TYP_MAX_EDIT_DISTANCE', 1))
    max_len_delta = int(globals().get('SYMSPELL_TYP_MAX_LEN_DELTA', 1))
    require_same_initial = bool(globals().get('SYMSPELL_TYP_REQUIRE_SAME_INITIAL', True))
    require_same_final = bool(globals().get('SYMSPELL_TYP_REQUIRE_SAME_FINAL', True))

    recovered_sym = symspell_recover_typo(
        token,
        allow_no_vowel=True,
        max_edit_distance=max_edit_distance,
        max_len_delta=max_len_delta,
    )
    if not recovered_sym:
        return None

    normalized = normalize_typo_token(token)
    recovered = recovered_sym['recovered']
    if require_same_initial and normalized and recovered and normalized[0] != recovered[0]:
        return None
    if require_same_final and len(normalized) >= 3 and recovered and normalized[-1] != recovered[-1]:
        return None

    formal_terms = globals().get('formal_terms') or set()
    if formal_terms and recovered not in formal_terms:
        return None
    return {
        'original': normalized,
        'matched_slang': None,
        'recovered': recovered,
        'source': 'symspell',
        'symspell': recovered_sym,
    }


def acoustic_switchboard(token: str, label: str, token_meta: dict | None = None) -> dict:
    """Switch board fonem berbasis label IndoBERT + lexicon slang.csv."""
    norm = label.upper().strip()
    cleaned = normalize_typo_token(token)

    if norm in LABELS_SINGKATAN:
        recovered = recover_formal_from_slang(token)
        if recovered:
            return {
                "switch_route": "SKT_SLANG_EXACT_G2P_ID",
                "switch_input": recovered['recovered'],
                "phoneme": g2p_id_phoneme(recovered['recovered']),
                "bc_info": {
                    "rule_used": "SLANG_EXACT",
                    "normalized_token": recovered['recovered'],
                    "recover": recovered,
                },
            }

        ejaan = singkatan_to_ejaan(token)
        return {
            "switch_route": "A_SINGKATAN_G2P_ID",
            "switch_input": ejaan,
            "phoneme": rumus_a_singkatan_simple(token),
        }

    if norm in LABELS_ASING:
        return choose_asg_phoneme(token)

    if norm in LABELS_DISEMVOWEL:
        recovered = recover_formal_from_slang(token)
        if recovered:
            return {
                "switch_route": "DSM_SLANG_EXACT_G2P_ID",
                "switch_input": recovered['recovered'],
                "phoneme": g2p_id_phoneme(recovered['recovered']),
                "bc_info": {
                    "rule_used": "SLANG_EXACT",
                    "normalized_token": recovered['recovered'],
                    "recover": recovered,
                },
            }

        return {
            "switch_route": "DSM_G2P_ID_DIRECT",
            "switch_input": cleaned,
            "phoneme": g2p_id_phoneme(cleaned),
            "bc_info": {
                "rule_used": "DIRECT",
                "normalized_token": cleaned,
            },
        }

    if norm in LABELS_TYPO:
        recovered = recover_formal_from_slang(token)
        if recovered:
            return {
                "switch_route": "TYP_SLANG_EXACT_G2P_ID",
                "switch_input": recovered['recovered'],
                "phoneme": g2p_id_phoneme(recovered['recovered']),
                "bc_info": {
                    "rule_used": "SLANG_EXACT",
                    "normalized_token": recovered['recovered'],
                    "recover": recovered,
                },
            }

        return {
            "switch_route": "TYP_G2P_ID_DIRECT",
            "switch_input": cleaned,
            "phoneme": g2p_id_phoneme(cleaned),
            "bc_info": {
                "rule_used": "DIRECT",
                "normalized_token": cleaned,
            },
        }

    return {
        "switch_route": "O_G2P_ID",
        "switch_input": cleaned,
        "phoneme": g2p_id_phoneme(cleaned),
    }

def _decode_bi_to_syllables(units: list[str], bi_pattern: str | None) -> dict:
    if not units:
        return {
            'valid': False,
            'labels': [],
            'syllables_units': [],
            'syllables_text': [],
            'error': 'units_empty',
        }

    labels = list((bi_pattern or '').strip())
    if len(labels) != len(units) or any(lb not in {'B', 'I'} for lb in labels):
        syll_text = ''.join(units)
        return {
            'valid': False,
            'labels': labels,
            'syllables_units': [units],
            'syllables_text': [syll_text],
            'error': 'invalid_bi_pattern',
        }

    syllables_units: list[list[str]] = []
    current: list[str] = []
    for i, (u, lb) in enumerate(zip(units, labels)):
        if i == 0 or lb == 'B':
            if current:
                syllables_units.append(current)
            current = [u]
        else:
            current.append(u)
    if current:
        syllables_units.append(current)

    syllables_text = [''.join(parts) for parts in syllables_units]
    return {
        'valid': True,
        'labels': labels,
        'syllables_units': syllables_units,
        'syllables_text': syllables_text,
        'error': None,
    }


def _is_vowel_unit_for_nucleus(unit: str) -> bool:
    if not unit:
        return False
    if _crf3_is_vowel(unit):
        return True
    if unit in {'eɪ', 'aɪ', 'aʊ', 'ɔɪ', 'oʊ'}:
        return True
    return any(ch in CRF3_VOWELS for ch in unit)


def _canonical_rhyme_unit(unit: str) -> str:
    if unit in {'ɪ', 'i'}:
        return 'i'
    if unit in {'ɛ', 'e'}:
        return 'e'
    if unit in {'ʊ', 'u'}:
        return 'u'
    if unit in {'ɔ', 'o', 'oʊ'}:
        return 'o'
    return unit


def _extract_rhyme_tail(syllable_text: str) -> str:
    if not syllable_text:
        return ''
    chars = [ch for ch in syllable_text if not ch.isspace()]
    if not chars:
        return ''
    for i in range(len(chars) - 1, -1, -1):
        if chars[i] in CRF3_VOWELS:
            return ''.join(chars[i:])
    return ''.join(chars)


def _extract_rhyme_tail_from_phoneme(phoneme: str, token: str = '') -> str:
    units = phoneme_to_units(phoneme)
    if not units:
        return ''

    if len(units) >= 2:
        u1 = units[-2]
        u2 = units[-1]
        if _is_vowel_unit_for_nucleus(u1) and _is_vowel_unit_for_nucleus(u2):
            pair = (_canonical_rhyme_unit(u1), _canonical_rhyme_unit(u2))
            if pair == ('a', 'i'):
                return 'aɪ'
            if pair == ('a', 'u'):
                return 'aʊ'
            if pair == ('o', 'i'):
                return 'ɔɪ'

    for i in range(len(units) - 1, -1, -1):
        if _is_vowel_unit_for_nucleus(units[i]):
            return ''.join(_canonical_rhyme_unit(u) for u in units[i:])

    normalized = normalize_typo_token(token)
    return normalized or ''.join(units)


def _count_vowel_nuclei(units: list[str]) -> int:
    return sum(1 for u in units if _is_vowel_unit_for_nucleus(u))


def _segment_asg_phoneme_with_fonotaktik(token: str, phoneme: str, bi_pattern: str | None = None) -> dict:
    units = phoneme_to_units(phoneme)
    if not units:
        normalized = normalize_typo_token(token)
        fallback = [normalized or token] if (normalized or token) else []
        return {
            'syllables': fallback,
            'syllable_count': len(fallback),
            'bi': '',
            'bi_valid': False,
        }

    # Jika diakhiri klitik Indonesia ('ɲa', 'ku', 'mu') yang ditempelkan oleh choose_asg_phoneme,
    # pisahkan 2 unit terakhir sebagai 1 silabel mandiri agar tidak tergabung dengan koda Inggris.
    clitic_suffix = None
    if len(units) >= 4 and tuple(units[-2:]) in {('ɲ', 'a'), ('k', 'u'), ('m', 'u')}:
        t_norm = normalize_typo_token(token)
        if t_norm.endswith(('nya', 'ku', 'mu')):
            clitic_suffix = ''.join(units[-2:])
            units = units[:-2]

    syll_fn = globals().get('syllabify_en_ipa_robust')
    if callable(syll_fn):
        out = syll_fn(units)
        if isinstance(out, dict):
            sylls = list(out.get('syllables') or [''.join(units)])
            bi_str = out.get('bi') or ('B' + 'I' * (len(units) - 1))
            if clitic_suffix:
                sylls.append(clitic_suffix)
                bi_str += 'BI'
            return {
                'syllables': sylls,
                'syllable_count': len(sylls),
                'bi': bi_str,
                'bi_valid': True,
            }

    fallback = [''.join(units)] + ([clitic_suffix] if clitic_suffix else [])
    bi_fb = 'B' + ('I' * (len(units) - 1)) + ('BI' if clitic_suffix else '')
    return {
        'syllables': fallback,
        'syllable_count': len(fallback),
        'bi': bi_fb,
        'bi_valid': True,
    }

def analyze_line_prosody_full(text: str) -> dict:
    token_rows = predict_with_routing(text)
    analyzed_tokens = []
    line_syllables = []
    line_units = []
    line_bi_flat = []

    for row in token_rows:
        token_label = row.get('label')
        if token_label in LABELS_SINGKATAN:
            token_syllables = [u for u in singkatan_to_ejaan(row.get('token') or '').split() if u]
            line_syllables.extend(token_syllables)
            analyzed_tokens.append({
                **row,
                'units': [],
                'unit_count': 0,
                'bi_pattern': '',
                'bi_valid': True,
                'bi_error': None,
                'syllables': token_syllables,
                'syllable_count': len(token_syllables),
                'rhyme_tail_token': _extract_rhyme_tail(singkatan_to_ejaan(row.get('token') or '')),
            })
            continue

        phoneme = row.get('phoneme') or ''
        units = phoneme_to_units(phoneme)
        switch_route = str(row.get('switch_route', ''))
        if switch_route.startswith('ASG_'):
            token_for_count = row.get('switch_input') or row.get('token') or ''
            asg_seg = _segment_asg_phoneme_with_fonotaktik(token_for_count, phoneme, row.get('bi_pattern'))
            token_syllables = asg_seg.get('syllables') or []
            token_bi = asg_seg.get('bi') or row.get('bi_pattern') or ''
            decoded = _decode_bi_to_syllables(units, token_bi)

            line_syllables.extend(token_syllables)
            line_units.extend(units)
            if decoded['valid']:
                line_bi_flat.extend(decoded['labels'])
            analyzed_tokens.append({
                **row,
                'units': units,
                'unit_count': len(units),
                'bi_pattern': token_bi,
                'bi_valid': bool(asg_seg.get('bi_valid')) and decoded['valid'],
                'bi_error': decoded['error'],
                'syllables': token_syllables,
                'syllable_count': len(token_syllables),
                'rhyme_tail_token': _extract_rhyme_tail_from_phoneme(phoneme, token=row.get('token') or ''),
            })
            continue

        bi_pattern = row.get('bi_pattern')
        if not bi_pattern:
            bi_pattern = predict_bi_from_phoneme(phoneme, token=row.get('token') or '')

        decoded = _decode_bi_to_syllables(units, bi_pattern)
        token_syllables = decoded['syllables_text']

        line_syllables.extend(token_syllables)
        line_units.extend(units)
        if decoded['valid']:
            line_bi_flat.extend(decoded['labels'])

        analyzed = {
            **row,
            'units': units,
            'unit_count': len(units),
            'bi_pattern': bi_pattern,
            'bi_valid': decoded['valid'],
            'bi_error': decoded['error'],
            'syllables': token_syllables,
            'syllable_count': len(token_syllables),
            'rhyme_tail_token': _extract_rhyme_tail_from_phoneme(phoneme, token=row.get('token') or ''),
        }
        analyzed_tokens.append(analyzed)

    total_syllables = len(line_syllables)
    final_syllable = line_syllables[-1] if line_syllables else ''
    final_rhyme = analyzed_tokens[-1].get('rhyme_tail_token', '') if analyzed_tokens else ''

    return {
        'input_text': text,
        'tokens': analyzed_tokens,
        'line_units': line_units,
        'line_unit_count': len(line_units),
        'line_bi_pattern_flat': ''.join(line_bi_flat),
        'line_syllables': line_syllables,
        'line_syllable_count': total_syllables,
        'final_syllable': final_syllable,
        'final_rhyme': final_rhyme,
    }


def print_line_prosody_analysis(report: dict) -> None:
    # print('=== ANALISIS LENGKAP LAYER-1 -> LAYER-3 ===')
    print(f"Input text         : {report['input_text']}")
    print(f"Total token        : {len(report['tokens'])}")
    print()

    # print('--- Detail per token ---')
    for idx, row in enumerate(report['tokens'], start=1):
        conf = row.get('model_confidence')
        margin = row.get('model_margin')
        conf_txt = f"{conf:.3f}" if conf is not None else '-'
        margin_txt = f"{margin:.3f}" if margin is not None else '-'

        print(
            f"[{idx:02d}] token={row['token']} | raw={row['raw_label']} | final={row['label']} | "
            f"route={row['route']} | switch={row['switch_route']} | conf={conf_txt} margin={margin_txt}"
        )
        if row.get('bc_info'):
            bc = row['bc_info']
            print(f"     bc_rule={bc.get('rule_used')} | normalized={bc.get('normalized_token')}")
        print(f"     phoneme={row.get('phoneme', '')}")
        print(f"     units={row['units']} (n={row['unit_count']})")
        print(
            f"     bi={row.get('bi_pattern') or '-'} | bi_valid={row.get('bi_valid')}"
            + (f" | bi_error={row.get('bi_error')}" if row.get('bi_error') else '')
        )
        print(f"     syllables={row.get('syllables', [])} | syllable_count={row.get('syllable_count', 0)}")
        print(f"     token_rhyme_tail={row.get('rhyme_tail_token', '')}")

    # print('\n--- Agregasi baris ---')
    # print(f"line_units          : {report['line_units']}")
    # print(f"line_unit_count     : {report['line_unit_count']}")
    # print(f"line_bi_pattern     : {report['line_bi_pattern_flat']}")
    # print(f"line_syllables      : {report['line_syllables']}")
    # print(f"JUMLAH_SILABEL      : {report['line_syllable_count']}")
    # print(f"SILABEL_AKHIR       : {report['final_syllable']}")
    # print(f"RIMA_AKHIR          : {report['final_rhyme']}")

# Pemetaan penyetaraan simbol vokal IPA Inggris (ARPAbet -> IPA)
# ke vokal dasar Indonesia untuk rima campur kode.
CROSS_LINGUAL_IPA_MAP = {
    'ʌ': 'a',  # corona (nʌ) -> na
    'ɑ': 'a',  # lockdown (lɑk) -> lak
    'æ': 'a',  # cat (kæt) -> kat
    'ɪ': 'i',  # matic (tɪk) -> tik
    'ʊ': 'u',  # book (bʊk) -> buk
    'ɛ': 'e',  # presiden (prɛ) -> pre
    'ɔ': 'o',  # boy (bɔɪ) -> boi
    'ə': 'e',  # schwa
    'ɝ': 'er',  # r-colored schwa Inggris (twitter [tɝ] -> ter -> rima 'er')
    'ʔ': 'k',  # hentian glotal akhir bahasa Indonesia (cantiʔ -> cantik) setara dengan koda /k/
}


def normalize_acoustic_rhyme_tail(rhyme_tail: str) -> str:
    t = (rhyme_tail or '').strip().lower()
    if not t:
        return ''
    return ''.join(CROSS_LINGUAL_IPA_MAP.get(ch, ch) for ch in t)


def _rhyme_key(rhyme_tail: str) -> str:
    norm = normalize_acoustic_rhyme_tail(rhyme_tail)
    if not norm:
        return '-'
    return norm[-2:] if len(norm) >= 2 else norm


def infer_rhyme_scheme(line_reports: list[dict]) -> str:
    keys = [_rhyme_key(rep.get('final_rhyme', '')) for rep in line_reports]
    mapping = {}
    next_label_ord = ord('a')
    scheme = []
    for key in keys:
        if key not in mapping:
            mapping[key] = chr(next_label_ord)
            next_label_ord += 1
        scheme.append(mapping[key])
    return '-'.join(scheme)


def analyze_pantun_multiline(text: str) -> dict:
    lines = [ln.strip() for ln in str(text).splitlines() if ln.strip()]
    line_reports = []
    for i, line in enumerate(lines, start=1):
        rep = analyze_line_prosody_full(line)
        rep['line_no'] = i
        line_reports.append(rep)

    rhyme_scheme = infer_rhyme_scheme(line_reports)
    total_syllables = sum(rep.get('line_syllable_count', 0) for rep in line_reports)

    return {
        'raw_text': text,
        'line_count': len(line_reports),
        'lines': line_reports,
        'total_syllables_all_lines': total_syllables,
        'rhyme_scheme': rhyme_scheme,
    }


def build_pantun_compact_output(report: dict) -> dict:
    lines = report.get('lines', [])

    compact = {
        'skema_rima': report.get('rhyme_scheme', ''),
        'rima_akhir_baris_1': '',
        'rima_akhir_baris_2': '',
        'rima_akhir_baris_3': '',
        'rima_akhir_baris_4': '',
        'suku_kata_baris_1': '',
        'suku_kata_baris_2': '',
        'suku_kata_baris_3': '',
        'suku_kata_baris_4': '',
    }

    for i in range(min(4, len(lines))):
        rep = lines[i]
        compact[f'rima_akhir_baris_{i+1}'] = normalize_acoustic_rhyme_tail(rep.get('final_rhyme', ''))
        compact[f'suku_kata_baris_{i+1}'] = rep.get('line_syllable_count', '')

    return compact


def print_pantun_multiline_analysis(report: dict, verbose: bool = False) -> None:
    compact = build_pantun_compact_output(report)
    cols = [
        'skema_rima',
        'rima_akhir_baris_1',
        'rima_akhir_baris_2',
        'rima_akhir_baris_3',
        'rima_akhir_baris_4',
        'suku_kata_baris_1',
        'suku_kata_baris_2',
        'suku_kata_baris_3',
        'suku_kata_baris_4',
    ]

    # print(','.join(str(compact[col]) for col in cols))

    if not verbose:
        return

    # print('\n=== ANALISIS PANTUN MULTI-BARIS (VERBOSE) ===')
    print(f"Jumlah baris terdeteksi : {report['line_count']}")
    print(f"Total silabel (semua)   : {report['total_syllables_all_lines']}")
    print(f"Skema rima              : {report['rhyme_scheme']}")

    for line_rep in report['lines']:
        print('\n' + '=' * 64)
        print(f"BARIS {line_rep['line_no']}: {line_rep['input_text']}")
        print_line_prosody_analysis(line_rep)





In [ ]:
# Contoh uji input pantun multi-baris (pisahkan dengan newline "\n")
sample_pantun = """pergi ke kota membawa uang\nbeli oleholeh cinderamata\nanak muda jaman sekarang\ntidak bisa lepas dari ponselnya"""

pantun_report = analyze_pantun_multiline(sample_pantun)

# Default: output ringkas sesuai format dataset
# print_pantun_multiline_analysis(pantun_report)

# Verbose (opsional): tampilkan semua detail per token/baris
print_pantun_multiline_analysis(pantun_report, verbose=True)

## Data training ByT5: teks → silabel grafem

Jalankan cell setup dan helper terlebih dahulu, lalu sesuaikan lokasi file di Drive. Centang `BUAT_DATA_BYT5` untuk membuat data; secara default proses **OFF**, termasuk saat menjalankan semua cell.

`text_pantun` dipisah menjadi satu contoh per baris, baik pemisah `\n` literal maupun newline sebenarnya. Progress menunjukkan jumlah pantun selesai, baris yang sedang diproses, waktu, dan jumlah baris yang perlu review. Lima kendala pertama ditampilkan saat proses berjalan; daftar lengkap tersimpan di file review setelah proses selesai. Jika terjadi error, proses berhenti dan traceback ditampilkan; CSV hanya ditulis setelah semua baris selesai dianalisis. File utama berkolom `text,silabel`.

Target merupakan **pseudo-label**, bukan anotasi kebenaran. Batas silabel fonem diproyeksikan secara perkiraan ke ejaan asli; slang, singkatan, kata asing, dan kesalahan model dapat menghasilkan batas keliru. Periksa file review dan audit sampel lainnya sebelum training: kecocokan grafem–fonem tidak menjamin batas silabel benar.

In [ ]:
# @title Buat CSV data ByT5 (opsional)
BUAT_DATA_BYT5 = False # @param {type:"boolean"}
TIMPA_OUTPUT_BYT5 = True # @param {type:"boolean"}
INPUT_BYT5_CSV = '/content/drive/MyDrive/pantun_gaul.csv' # @param {type:"string"}
OUTPUT_BYT5_CSV = '/content/drive/MyDrive/Scripts/pantun_byt5.csv' # @param {type:"string"}
REVIEW_BYT5_CSV = '/content/drive/MyDrive/Scripts/pantun_byt5_review.csv' # @param {type:"string"}

if BUAT_DATA_BYT5:
    import csv
    import os
    from difflib import SequenceMatcher
    from tqdm.auto import tqdm

    if not callable(globals().get('analyze_line_prosody_full')) or not callable(globals().get('_clean_input_words')):
        raise RuntimeError('Jalankan cell setup dan helper notebook terlebih dahulu.')
    if globals().get('crf_bi_model') is None:
        raise RuntimeError('Model CRF layer-3 belum dimuat; periksa CRF_BI_MODEL_PATH.')

    input_path = os.path.realpath(INPUT_BYT5_CSV)
    output_path = os.path.realpath(OUTPUT_BYT5_CSV)
    review_path = os.path.realpath(REVIEW_BYT5_CSV)
    if len({input_path, output_path, review_path}) != 3:
        raise ValueError('Path input, output, dan review harus berbeda.')
    for path in (output_path, review_path):
        if os.path.exists(path) and not TIMPA_OUTPUT_BYT5:
            raise FileExistsError(f'File sudah ada: {path}. Aktifkan TIMPA_OUTPUT_BYT5 jika ingin menimpa.')

    # Pendekatan grafem umum, bukan konversi IPA yang dapat dibalik secara pasti.
    ipa_to_grafem = str.maketrans({
        'ŋ': 'ng', 'ɲ': 'ny', 'ʃ': 'sy', 'ɡ': 'g', 'ʔ': 'k',
        'ə': 'e', 'ɛ': 'e', 'ɪ': 'i', 'ʊ': 'u', 'ɔ': 'o',
        'ɑ': 'a', 'æ': 'a', 'ʌ': 'a', 'ɒ': 'o', 'ɐ': 'a',
        'ˈ': '', 'ˌ': '', 'ː': '',
    })

    def proyeksi_silabel(token, syllables):
        parts = [str(s).lower().translate(ipa_to_grafem) for s in syllables]
        fonem = ''.join(parts)
        grafem = token.lower()
        if not fonem or not parts:
            return token, ['silabel fonem kosong']
        matcher = SequenceMatcher(None, fonem, grafem, autojunk=False)
        opcodes = matcher.get_opcodes()
        alasan = [] if fonem == grafem else ['grafem dan fonem tidak identik']
        if ''.join(str(s).lower() for s in syllables) != fonem:
            alasan.append('konversi simbol IPA perkiraan')
        batas_fonem = 0
        batas_grafem = []
        for part in parts[:-1]:
            batas_fonem += len(part)
            posisi = None
            for tag, a1, a2, b1, b2 in opcodes:
                if a1 <= batas_fonem <= a2 and a2 > a1:
                    posisi = b1 + round((batas_fonem - a1) * (b2 - b1) / (a2 - a1))
                    break
            if posisi is None or posisi <= 0 or posisi >= len(token) or (batas_grafem and posisi <= batas_grafem[-1]):
                alasan.append('batas silabel tidak dapat diproyeksikan')
            else:
                batas_grafem.append(posisi)
        hasil = []
        awal = 0
        for akhir in batas_grafem + [len(token)]:
            hasil.append(token[awal:akhir])
            awal = akhir
        return '-'.join(hasil), alasan

    def silabel_baris(text):
        rows = analyze_line_prosody_full(text)['tokens']
        words = _clean_input_words(text)
        if [row['token'] for row in rows] != words:
            raise ValueError(f'Token hasil model tidak lengkap untuk baris: {text!r}')
        hasil, alasan = [], []
        if not words:
            alasan.append('tidak ada token untuk dianalisis')
        posisi = 0
        for row in rows:
            token = row['token']
            awal = text.find(token, posisi)
            if awal < 0:
                raise ValueError(f'Token {token!r} tidak ditemukan dalam baris {text!r}')
            hasil.append(text[posisi:awal])
            segmentasi, masalah = proyeksi_silabel(token, row.get('syllables') or [])
            if row.get('label') != 'O':
                masalah.append('label ' + str(row.get('label')))
            if not row.get('bi_valid', False):
                masalah.append('pola BI tidak valid')
            if str(row.get('switch_input') or '').lower() != token.lower():
                masalah.append('input fonem berbeda dari token asli')
            if masalah:
                alasan.append(f"{token}: {', '.join(masalah)}")
            hasil.append(segmentasi)
            posisi = awal + len(token)
        hasil.append(text[posisi:])
        return ''.join(hasil), '; '.join(alasan)

    print(f'Membaca input: {input_path}', flush=True)
    data, review = [], []
    with open(input_path, encoding='utf-8-sig', newline='') as sumber:
        reader = csv.DictReader(sumber)
        if not reader.fieldnames or 'text_pantun' not in reader.fieldnames:
            raise ValueError('CSV input harus memiliki kolom text_pantun.')
        pantun_rows = list(reader)

    print(f'Mulai analisis {len(pantun_rows)} pantun. Output disimpan setelah analisis selesai.', flush=True)
    with tqdm(total=len(pantun_rows), desc='Pantun', unit='pantun') as progress:
        for nomor_csv, row in enumerate(pantun_rows, start=2):
            progress.set_postfix(csv=nomor_csv, baris='-', data=len(data), review=len(review))
            pantun = row.get('text_pantun')
            if pantun is None or not pantun.strip():
                raise ValueError(f'text_pantun kosong pada baris CSV {nomor_csv}.')
            for nomor_baris, text in enumerate(pantun.replace('\\n', '\n').splitlines(), start=1):
                text = text.strip()
                if not text:
                    continue
                progress.set_postfix(csv=nomor_csv, baris=nomor_baris, data=len(data), review=len(review), teks=text[:60])
                silabel, alasan = silabel_baris(text)
                data.append({'text': text, 'silabel': silabel})
                if alasan:
                    review.append({'baris_csv': nomor_csv, 'baris_pantun': nomor_baris, 'text': text, 'silabel': silabel, 'alasan': alasan})
                    if len(review) <= 5:
                        tqdm.write(f'[Review] CSV {nomor_csv}, baris pantun {nomor_baris}: {alasan}')
                    elif len(review) == 6:
                        tqdm.write('[Review] Kendala berikutnya dicatat di file review setelah proses selesai.')
                progress.set_postfix(csv=nomor_csv, baris=nomor_baris, data=len(data), review=len(review), teks=text[:60])
            progress.update(1)

    if not data:
        raise ValueError('Tidak ada baris pantun untuk diekspor.')
    for path, fields, records in (
        (output_path, ['text', 'silabel'], data),
        (review_path, ['baris_csv', 'baris_pantun', 'text', 'silabel', 'alasan'], review),
    ):
        print(f'Menyimpan: {path}', flush=True)
        with open(path, 'w', encoding='utf-8', newline='') as output:
            writer = csv.DictWriter(output, fieldnames=fields)
            writer.writeheader()
            writer.writerows(records)
    print(f'Data: {len(data)} baris → {output_path}')
    print(f'Perlu review: {len(review)} baris → {review_path}')
else:
    print('Ekspor ByT5 OFF. Aktifkan BUAT_DATA_BYT5 untuk memulai.')


# Pembuatan Golden Annotation untuk Pantun Gaul menggunakan IndoBERT + CRF dan perbandingan dengan utils.py

Hasilnya akan disimpan dalam format CSV yang memuat status review, detail perbedaan, pemenggalan suku kata, skema rima, silabel, dan rima akhir per baris.

Script ini memanfaatkan fungsi analyze_pantun_multiline dan build_pantun_compact_output dari helper IndoBERT + CRF untuk melakukan analisis otomatis, kemudian membandingkannya dengan hasil utils.py agar dapat menghasilkan golden annotation yang dapat direview secara manual.

In [ ]:
import re
import pandas as pd
from tqdm.auto import tqdm

# ==============================================================================
# KONFIGURASI FILE INPUT & OUTPUT GOLDEN ANNOTATION
# ==============================================================================
# Cara pakai:
# - False: cell aman saat Run All (tidak membaca/menulis file).
# - True : proses golden annotation dijalankan dan hasil disimpan ke OUTPUT_REVIEW_CSV.
RUN_GOLDEN_ANNOTATION = False  # OFF by default: ubah ke True untuk mengeksekusi proses
INPUT_CSV_PATH = '/content/drive/MyDrive/pantun_gaul.csv'
OUTPUT_REVIEW_CSV = '/content/drive/MyDrive/pantun_golden_review.csv'

# Pastikan fungsi dari cell helper sudah dimuat (hanya saat RUN = True)
if RUN_GOLDEN_ANNOTATION and (not callable(globals().get('analyze_pantun_multiline')) or not callable(globals().get('build_pantun_compact_output'))):
    raise RuntimeError('Jalankan cell setup dan helper IndoBERT + CRF terlebih dahulu.')

if RUN_GOLDEN_ANNOTATION:
    df = pd.read_csv(INPUT_CSV_PATH)
else:
    df = pd.DataFrame(columns=['text_pantun'])
    print('[GOLDEN_ANNOTATION] OFF: set RUN_GOLDEN_ANNOTATION=True lalu jalankan ulang cell untuk memproses.')

if RUN_GOLDEN_ANNOTATION and 'text_pantun' not in df.columns:
    raise ValueError(f"Kolom 'text_pantun' tidak ditemukan. Kolom tersedia: {list(df.columns)}")


def _find_col(df_cols, candidates):
    lower_map = {c.lower().strip(): c for c in df_cols}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    return None


def _norm_rhyme_for_compare(val) -> str:
    """Normalisasi IPA ke grafem agar perbedaan penulisan (mis. 'aɪ' vs 'ai') tidak dianggap konflik palsu."""
    s = str(val or '').strip().lower()
    if s in {'', 'nan', 'none'}:
        return ''
    replacements = [
        ('aɪ', 'ai'), ('aʊ', 'au'), ('ɔɪ', 'oi'), ('eɪ', 'ei'), ('oʊ', 'ou'),
        ('ŋ', 'ng'), ('ɲ', 'ny'), ('ʃ', 'sy'), ('ʔ', 'k'),
        ('ə', 'e'), ('ɛ', 'e'), ('ɪ', 'i'), ('ʊ', 'u'), ('ɔ', 'o'), ('ɑ', 'a'),
    ]
    for src, dst in replacements:
        s = s.replace(src, dst)
    return s


def _safe_int(val):
    try:
        if pd.isna(val) or str(val).strip() == '':
            return None
        return int(float(val))
    except Exception:
        return None


# Deteksi otomatis nama kolom hasil utils.py di pantun_gaul.csv
col_u_skema = _find_col(df.columns, ['skema_rima', 'rhyme_scheme', 'utils_skema_rima'])
cols_u_rima = [
    _find_col(df.columns, [f'rima_akhir_baris_{i}', f'rima_baris_{i}', f'rima_{i}'])
    for i in range(1, 5)
]
cols_u_sil = [
    _find_col(df.columns, [f'suku_kata_baris_{i}', f'silabel_baris_{i}', f'silabel_{i}', f'jumlah_silabel_{i}'])
    for i in range(1, 5)
]

print('Kolom utils terdeteksi:')
print(' - Skema rima :', col_u_skema)
print(' - Rima baris :', cols_u_rima)
print(' - Silabel    :', cols_u_sil)

# ==============================================================================
# INFERENSI INDOBERT + CRF & KOMPARASI BERSEBELAHAN
# ==============================================================================
review_rows = []
cocok_count = 0
beda_count = 0

for idx, row in tqdm(df.iterrows(), total=len(df), desc='Proses Golden Annotation'):
    raw_pantun = str(row.get('text_pantun') or '').replace('\\n', '\n').strip()
    report = analyze_pantun_multiline(raw_pantun)
    compact = build_pantun_compact_output(report)

    # Susun pemenggalan suku kata per baris agar mudah dicek secara visual
    pemenggalan_baris = []
    for ln in report.get('lines', []):
        tok_str = ' '.join('-'.join(t.get('syllables', [])) for t in ln.get('tokens', []))
        pemenggalan_baris.append(f"B{ln['line_no']}: {tok_str} ({ln['line_syllable_count']})")

    diffs = []

    # 1. Bandingkan Skema Rima
    u_skema = str(row[col_u_skema]).strip() if col_u_skema and pd.notna(row[col_u_skema]) else ''
    ib_skema = str(compact.get('skema_rima') or '').strip()
    if u_skema and ib_skema and u_skema != ib_skema:
        diffs.append(f"Skema Rima (utils={u_skema} vs ib={ib_skema})")

    # 2. Bandingkan Jumlah Silabel & Rima Akhir per Baris (1..4)
    u_sil_vals, ib_sil_vals = [], []
    u_rima_vals, ib_rima_vals = [], []

    for i in range(1, 5):
        u_s = _safe_int(row[cols_u_sil[i - 1]]) if cols_u_sil[i - 1] else None
        ib_s = _safe_int(compact.get(f'suku_kata_baris_{i}'))
        u_sil_vals.append(u_s if u_s is not None else '')
        ib_sil_vals.append(ib_s if ib_s is not None else '')

        if u_s is not None and ib_s is not None and u_s != ib_s:
            diffs.append(f"Silabel B{i} (utils={u_s} vs ib={ib_s})")

        u_r = str(row[cols_u_rima[i - 1]]).strip() if cols_u_rima[i - 1] and pd.notna(row[cols_u_rima[i - 1]]) else ''
        ib_r = str(compact.get(f'rima_akhir_baris_{i}') or '').strip()
        u_rima_vals.append(u_r)
        ib_rima_vals.append(ib_r)

        if u_r and ib_r and _norm_rhyme_for_compare(u_r) != _norm_rhyme_for_compare(ib_r):
            diffs.append(f"Rima B{i} (utils={u_r} vs ib={ib_r})")

    is_conflict = len(diffs) > 0
    if is_conflict:
        beda_count += 1
        status = 'PERLU_REVIEW'
    else:
        cocok_count += 1
        status = 'COCOK'

    out_record = {
        'no': idx + 1,
        'status_review': status,
        'detail_perbedaan': '; '.join(diffs) if diffs else '-',
        'text_pantun': raw_pantun,
        'indobert_pemenggalan': "\n".join(pemenggalan_baris),

        # --- Skema Rima Bersebelahan ---
        'utils_skema_rima': u_skema,
        'ib_skema_rima': ib_skema,
        'golden_skema_rima': '' if is_conflict else ib_skema,
    }

    # --- Silabel per Baris Bersebelahan ---
    for i in range(4):
        b = i + 1
        out_record[f'utils_silabel_b{b}'] = u_sil_vals[i]
        out_record[f'ib_silabel_b{b}'] = ib_sil_vals[i]
        out_record[f'golden_silabel_b{b}'] = '' if (u_sil_vals[i] != ib_sil_vals[i]) else ib_sil_vals[i]

    # --- Rima Akhir per Baris Bersebelahan ---
    for i in range(4):
        b = i + 1
        out_record[f'utils_rima_b{b}'] = u_rima_vals[i]
        out_record[f'ib_rima_b{b}'] = ib_rima_vals[i]
        same_rhyme = _norm_rhyme_for_compare(u_rima_vals[i]) == _norm_rhyme_for_compare(ib_rima_vals[i])
        out_record[f'golden_rima_b{b}'] = ib_rima_vals[i] if same_rhyme else ''

    review_rows.append(out_record)

if RUN_GOLDEN_ANNOTATION:
    df_review = pd.DataFrame(review_rows)
    # Simpan dengan utf-8-sig agar simbol fonetik IPA terbaca rapi di Excel / Google Sheets
    df_review.to_csv(OUTPUT_REVIEW_CSV, index=False, encoding='utf-8-sig')

    print('\n=== RINGKASAN PEMBUATAN GOLDEN ANNOTATION ===')
    print(f'Total pantun diproses : {len(df_review)}')
    print(f'COCOK (auto-filled)   : {cocok_count} ({cocok_count / len(df_review) * 100:.1f}%)')
    print(f'PERLU_REVIEW (beda)   : {beda_count} ({beda_count / len(df_review) * 100:.1f}%)')
    print(f'File tersimpan di     : {OUTPUT_REVIEW_CSV}')